TUGAS MODUL10

In [1]:
import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"
os.environ["PATH"] = os.environ["JAVA_HOME"] + "/bin:" + os.environ["PATH"]

import shutil, time
import pandas as pd
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, TimestampType
from pyspark.sql.functions import col, window, sum as spark_sum, count

spark = SparkSession.builder \
    .appName("Tugas10-KopiNusantara") \
    .master("local[*]") \
    .config("spark.hadoop.fs.defaultFS", "file:///") \
    .getOrCreate()
spark.sparkContext.setLogLevel("ERROR")
spark.conf.set("spark.sql.shuffle.partitions", "4")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/11/05 10:37:14 WARN Utils: Your hostname, vmi3356457, resolves to a loopback address: 127.0.1.1; using 13.140.149.59 instead (on interface eth0)
26/11/05 10:37:14 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
/home/zhio/jupyter-env/lib/python3.12/site-packages/pyspark/testing/utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
26/11/05 10:37:16 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [2]:
folder_stream = "tugas10_stream"
for f in (folder_stream, "tugas10_output", "tugas10_checkpoint"):
    shutil.rmtree(f, ignore_errors=True)
os.makedirs(folder_stream, exist_ok=True)

skema_pesanan = StructType([
    StructField("pesanan_id", StringType(), True),
    StructField("nama_cabang", StringType(), True),
    StructField("nama_menu", StringType(), True),
    StructField("qty", IntegerType(), True),
    StructField("harga_satuan", IntegerType(), True),
    StructField("waktu_pesan", TimestampType(), True),
])

df_stream = spark.readStream.format("csv") \
    .schema(skema_pesanan).option("header", True).load(folder_stream)

In [3]:
df_total = df_stream.withColumn("total_harga", col("qty") * col("harga_satuan"))

agregasi_stream = df_total.groupBy("nama_cabang").agg(
    spark_sum("total_harga").alias("total_pendapatan"),
    count("pesanan_id").alias("jumlah_pesanan")
)

query1 = agregasi_stream.writeStream \
    .outputMode("complete").format("memory").queryName("agregasi_cabang").start()

In [4]:
df_stream2 = spark.readStream.format("csv") \
    .schema(skema_pesanan).option("header", True).load(folder_stream) \
    .withColumn("total_harga", col("qty") * col("harga_satuan"))

agregasi_window = df_stream2.groupBy(window(col("waktu_pesan"), "2 minutes")).agg(
    spark_sum("total_harga").alias("total_pendapatan")
)

query2 = agregasi_window.writeStream \
    .outputMode("complete").format("memory").queryName("agregasi_waktu").start()

In [5]:
pd.DataFrame({
    "pesanan_id": ["P1", "P2", "P3"],
    "nama_cabang": ["Magelang", "Solo", "Magelang"],
    "nama_menu": ["Latte", "Americano", "Croissant"],
    "qty": [2, 1, 3],
    "harga_satuan": [25000, 20000, 18000],
    "waktu_pesan": ["2026-11-05 10:00:10", "2026-11-05 10:00:40", "2026-11-05 10:01:30"],
}).to_csv(f"{folder_stream}/batch_1.csv", index=False)

time.sleep(10)
spark.sql("SELECT * FROM agregasi_cabang ORDER BY nama_cabang").show()

+-----------+----------------+--------------+
|nama_cabang|total_pendapatan|jumlah_pesanan|
+-----------+----------------+--------------+
|   Magelang|          104000|             2|
|       Solo|           20000|             1|
+-----------+----------------+--------------+



In [6]:
query3 = df_stream.writeStream \
    .format("parquet") \
    .option("path", "tugas10_output") \
    .option("checkpointLocation", "tugas10_checkpoint") \
    .outputMode("append") \
    .start()
time.sleep(10)

In [7]:
spark.sql("SELECT window.start, window.end, total_pendapatan FROM agregasi_waktu ORDER BY window.start").show(truncate=False)

+-------------------+-------------------+----------------+
|start              |end                |total_pendapatan|
+-------------------+-------------------+----------------+
|2026-11-05 10:00:00|2026-11-05 10:02:00|124000          |
+-------------------+-------------------+----------------+

